In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim
import pyspark.sql.functions as F

df_procedures_bronze = spark.read.table("workspace.bronze.procedures")

df_procedures_silver = (
    df_procedures_bronze
    .filter(col("PATIENT").isNotNull())
    .filter(col("ENCOUNTER").isNotNull())
    .filter(col("DATE").isNotNull())
    .select(
        col("DATE").alias("date"),
        col("PATIENT").alias("patient_id"),
        col("ENCOUNTER").alias("encounter_id"),
        col("CODE").alias("code"),
        col("DESCRIPTION").alias("description"),
        col("REASONCODE").alias("reason_code"),
        col("REASONDESCRIPTION").alias("reason_description"),
        col("ingested_at")
    )
)

(
    df_procedures_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.procedures")
)

print(f"✅ Created workspace.silver.patients with {df_procedures_silver.count()} clean rows!")